In [1]:
import torch
from dinosaw.models.vit_wrapper import MODEL_LIST, PretrainedViTWrapper, AlibiVitWrapper
from dinosaw.utils import do_2D_pca, to_numpy, closest_resize, convert_image

import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import minmax_scale, scale
from os import listdir

from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib import font_manager


SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)

/home/ronan/Documents/phd/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
alibi_model = AlibiVitWrapper(
    MODEL_LIST[1],
    stride=14,
    add_flash_attn=False,
    device="cuda:0",
    slope_type="constant",
    normalize=True,
    wrap=True,
)

weights = torch.load("../../trained_models/alibi_dv2_vits14_reg.pth", weights_only=True, map_location="cuda:0")
alibi_model.load_state_dict(weights)
alibi_model.eval()
None

In [3]:
dv2_model = PretrainedViTWrapper(
    MODEL_LIST[1],
    stride=14,
    add_flash_attn=False,
    device="cuda:0",
)

dv2_model.eval()
None

In [4]:
def get_features(model: PretrainedViTWrapper, pil_img: Image.Image) -> np.ndarray:
    tr = closest_resize(pil_img.height, pil_img.width, 14)
    img_tensor = convert_image(pil_img, tr, device_str="cuda:0", to_half=False)
    with torch.no_grad():
        emb = model.forward_features(img_tensor, make_2D=True)
    emb_np = to_numpy(emb.squeeze(0))
    return emb_np

In [5]:
def add_custom_font(font_folder: str, font_name: str="Grotesk") -> None:
    try:
        font_paths = (f'{font_folder}/{font_name}.ttf', f'{font_folder}/{font_name}-Bold.ttf')
        for font_path in font_paths:
            font_manager.fontManager.addfont(font_path)
        prop = font_manager.FontProperties(fname=font_paths[0])

        plt.rcParams['font.family'] = 'sans-serif'
        plt.rcParams['font.sans-serif'] = prop.get_name()
    except Exception as e:
        print(f"Can't load custom font: {e} ")

In [6]:
def get_shared_pca(features_: list[np.ndarray], fg_masks: list[np.ndarray]) -> PCA:
    all_features = []
    for feature, fg_mask in zip(features_, fg_masks):
        masked = feature[fg_mask]
        all_features.append(masked)
        print(feature.shape, fg_mask.shape, masked.shape)
    all_features_concat = np.concatenate(all_features, axis=0)
    all_features_scaled = scale(all_features_concat, axis=0)
    pca = PCA(n_components=3)
    pca.fit(all_features_scaled)
    return pca

def apply_shared_pca(features: np.ndarray, fg_mask: np.ndarray, pca: PCA) -> np.ndarray:
    h, w, c = features.shape
    valid_features = features[fg_mask]
    features_scaled = scale(valid_features, axis=0)
    emb_3d = pca.transform(features_scaled)
    emb_3d_minmax = minmax_scale(emb_3d, feature_range=(0, 1), axis=0)

    out = np.zeros((h, w, 3), dtype=np.float32)
    out[fg_mask] = emb_3d_minmax
    return out

In [7]:
images: list[Image.Image] = []
path = 'data/shared_pca/'
folder_name = 'bison'

for file_name in listdir(f'{path}/{folder_name}'):
    img = Image.open(f'{path}/{folder_name}/{file_name}').convert('RGB')

    shortest = min(img.size)
    L = 384
    sf = L / shortest
    new_size = (int(img.size[0] * sf), int(img.size[1] * sf))
    img = img.resize(new_size, resample=Image.LANCZOS)

    images.append(img)

In [8]:
features = {'dv2': [], 'alibi': []}
features_reduced = {'dv2': [], 'alibi': []}
for img in images:
    for name, model in (('dv2', dv2_model), ('alibi', alibi_model)):
        feat = get_features(model, img)
        reduced = do_2D_pca(feat, 3, pre_norm='std', post_norm='minmax')
        features[name].append(feat)
        features_reduced[name].append(reduced)

In [9]:
def _get(red_li, img_idx, ch, thr) -> np.ndarray:
    return red_li[img_idx][: , :, ch] > thr

# bison
red = features_reduced['alibi']
fg_masks =[_get(red, 0, 1, 0.7), ~_get(red, 1, 0, 0.3), ~_get(red, 2, 0, 0.3), ~_get(red, 3, 0, 0.4)]

selected_masks = fg_masks

In [10]:
# # plane-birds
# dv2_fg_masks = [reduced_features[0][: , :, 1] > 0.7, reduced_features[1][: , :, 0] < 0.3, reduced_features[2][: , :, 0]  < 0.3, reduced_features[3][: , :, 0] > 0.6]
# alibi_fg_masks = [reduced_features[0][: , :, 0] < 0.5, reduced_features[1][: , :, 0] < 0.45, reduced_features[2][: , :, 0]  < 0.35, reduced_features[3][: , :, 0] < 0.4]

# selected_masks = alibi_fg_masks

In [11]:
# # elephants
# dv2_fg_masks = [reduced_features[0][: , :, 1] > 0.7, reduced_features[1][: , :, 0] < 0.3, reduced_features[2][: , :, 0]  < 0.3, reduced_features[3][: , :, 0] > 0.6]
# alibi_fg_masks = [reduced_features[0][: , :, 0] > 0.5, reduced_features[1][: , :, 0] > 0.45, reduced_features[2][: , :, 0]  > 0.35, reduced_features[3][: , :, 0] < 0.3]

# selected_masks = alibi_fg_masks

In [12]:
%%capture
# fig, axs = plt.subplots(len(images), 2, figsize=(12, 8))

# for i, (img, reduced, mask) in enumerate(zip(images, reduced_features, selected_masks)):
#     axs[i, 0].imshow(img)
#     axs[i, 0].axis('off')
#     axs[i, 0].set_title(f'Image {i+1}', fontsize=14)

#     axs[i, 1].imshow(reduced * mask[:, :, np.newaxis])
#     axs[i, 1].axis('off')
#     axs[i, 1].set_title(f'PCA Reduced Features {i+1}', fontsize=14)
# plt.tight_layout()

In [13]:
%%capture
dv2_feats_tr = [f.transpose(1, 2, 0) for f in features['dv2']]
dv2_shared_pca = get_shared_pca(dv2_feats_tr, fg_masks)

alibi_feats_tr = [f.transpose(1, 2, 0) for f in features['alibi']]
alibi_shared_pca = get_shared_pca(alibi_feats_tr, selected_masks)

dv2_fg_feats_reduced = [apply_shared_pca(feats, mask, dv2_shared_pca) for feats, mask in zip(dv2_feats_tr, selected_masks)]
alibi_fg_feats_reduced = [apply_shared_pca(feats, mask, alibi_shared_pca) for feats, mask in zip(alibi_feats_tr, selected_masks)]

In [ ]:
%%capture
W, H = 3, 2.5
FS = 28
add_custom_font('resources/fonts', 'Grotesk')
fig, axs = plt.subplots(len(images), 3, figsize=(10, H * 5))

for i, (img, dv2_reduced, alibi_reduced, mask) in enumerate(zip(images, dv2_fg_feats_reduced, alibi_fg_feats_reduced, selected_masks)):
    axs[i, 0].imshow(img)
    axs[i, 0].axis('off')

    axs[i, 1].imshow(dv2_reduced * mask[:, :, np.newaxis])
    axs[i, 1].axis('off')
    

    axs[i, 2].imshow(alibi_reduced * mask[:, :, np.newaxis])
    axs[i, 2].axis('off')

    if i == 0:
        axs[i, 1].set_title('DINOv2', fontsize=FS)
        axs[i, 2].set_title('ALiBi-Dv2', fontsize=FS)
plt.tight_layout()
plt.savefig('saved/03b.png', dpi=300)